# Protenix Structure Prediction

This notebook demonstrates multi-modal structure prediction using Protenix, an open-source reimplementation of AlphaFold3 developed by ByteDance Research. Protenix predicts the 3D structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes. The example here predicts a protein-ligand complex: the MfnG protein bound to L-tyrosine. It covers input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("protenix")
display_overview("protenix")
display_docs_section("protenix", "Background")

# Protenix

> Protenix is ByteDance's open-source reimplementation of AlphaFold3 that predicts 3D structures of biomolecular complexes using a [diffusion](https://en.wikipedia.org/wiki/Diffusion_model)-based architecture. It supports proteins, DNA, RNA, ligands, and their multi-chain complexes with optional post-translational modifications. Protenix is the first fully open-source model to match or exceed AlphaFold3 accuracy across diverse benchmarks.

## Background

Predicting the 3D structure of biomolecular complexes is fundamental to understanding biological function. Protein-protein interactions govern signaling pathways, protein-DNA interactions control gene regulation, and protein-ligand binding underlies drug action.

Protenix uses a diffusion-based architecture (following the AlphaFold3 approach) that:

* **Generates full-atom coordinates** rather than inter-residue distances, enabling direct prediction of ligand binding poses and nucleic acid conformations
* **Handles multi-modal inputs** natively: proteins, DNA, RNA, and small molecule ligands in a single prediction
* **Supports chemical modifications** via [Chemical Component Dictionary](https://www.wwpdb.org/data/ccd) (CCD) codes, enabling prediction of structures with phosphorylation, methylation, and other PTMs
* **Uses MSA information** (optional) from ColabFold search to capture evolutionary conservation signals

The model architecture consists of a Pairformer module (iterative refinement of pair representations), followed by a diffusion module that denoises random 3D coordinates into the predicted structure. Multiple samples are generated and ranked by a confidence head.

## Available tools

In [2]:
display_available_tools("protenix")

- **`run_protenix()`** — Multi-modal structure prediction using Protenix (open-source AlphaFold3)

### `run_protenix`

Protenix is ByteDance Research's open-source reimplementation of AlphaFold3. It supports protein, DNA, RNA, and ligand inputs in a single unified model. For each complex, Protenix runs multiple diffusion samples and returns the highest-confidence prediction according to a ranking score that combines pTM, ipTM, pLDDT, and global predicted distance error. The base model with MSA integration delivers production-quality accuracy comparable to the original AlphaFold3, while mini and tiny variants offer faster inference for screening workflows.

In [3]:
from pathlib import Path

from proto_tools import (
    Chain,
    ProtenixConfig,
    ProtenixInput,
    StructurePredictionComplex,
    run_protenix,
)

In [4]:
# Display input API reference
display_api_reference("protenix", "input", "run_protenix")

# MfnG protein sequence (384 residues)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = ProtenixInput(complexes=[complex])

**Input** — `ProtenixInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | required | List of complexes to predict structures for. Inherited from `StructurePredictionInput`. Each complex can contain multiple chains of proteins, DNA, RNA, and/or ligands. |
| `chains` | `List[Chain]` | required | Chains in the complex. Each chain can be: |
| `msas` | `Dict[string, MSA]` |  | Pre-computed MSAs keyed by protein sequence. Populated by preprocess() or supplied directly. Default: None. |

In [5]:
# Display config API reference
display_api_reference("protenix", "config", "run_protenix")

# Configure Protenix using the base model with MSA integration
config = ProtenixConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    model_name="protenix_base_default_v1.0.0",
    use_msa=True,
    num_pairformer_cycles=10,
    num_diffusion_samples=5,
    num_diffusion_steps=200,
    seeds=[0],
)

**Config** — `ProtenixConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_name` | `enum` | `protenix_base_default_v1.0.0` | Protenix model variant to use. Available models: Available options: `protenix_base_default_v1.0.0`, `protenix_base_20250630_v1.0.0`, `protenix_base_default_v0.5.0`, `protenix_base_constraint_v0.5.0`, `protenix_mini_esm_v0.5.0`, `protenix_mini_ism_v0.5.0`, `protenix_mini_default_v0.5.0`, `protenix_tiny_default_v0.5.0` |
| `seeds` | `List[integer]` | `[0]` | Random seeds for structure sampling. Each seed produces `num_diffusion_samples` independent structure samples. Multiple seeds increase diversity of the sampled conformations. A single seed is sufficient for most use cases; more seeds may help for challenging docking tasks such as antibody-antigen complexes. |
| `num_diffusion_samples` | `integer` | `5` | Number of independent structure samples to generate per seed. Only the best sample (by ranking score) is returned. Higher values explore more of the conformational space but increase computation time. Must be at least 1. Default: 5. |
| `num_diffusion_steps` | `integer` | `200` | Number of denoising steps in the diffusion process. Higher values produce more refined structures but are slower. Typical range: 100-500. Must be at least 1. Default: 200. |
| `num_pairformer_cycles` | `integer` | `10` | Number of Pairformer recycling iterations through the model. Higher values produce more refined structures but increase computation time. Typical range: 3-20. Must be at least 0. Default: 10. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `use_msa` | `boolean` | `True` | Whether to generate and use Multiple Sequence Alignments (MSAs) for protein chains using ColabFold search. Inherited from `MSAStructurePredictionConfig`. Default: `True`. |
| `verbose` | `boolean` | `False` | Whether to print status messages during execution including MSA generation, model loading, and prediction progress. Inherited from `StructurePredictionConfig`. Default: `False`. |
| `device` | `string` | `cuda` | Device to run the model on (e.g., `"cuda"`, `"cpu"`). Inherited from `StructurePredictionConfig`. Default: `"cuda"`. |
| `timeout` | `integer` | `1200` | Maximum execution time in seconds. Base models need \~10-15 minutes on slower GPUs. Default: 1200. |
| `colabfold_search_config` | `ColabfoldSearchConfig` |  | Configuration for ColabFold MSA search. Only used when `use_msa=True`. Inherited from `MSAStructurePredictionConfig`. Default: `None`. |

In [6]:
# Run the prediction
result = run_protenix(inputs, config)

Running run_colabfold_search [00:00]

Running Protenix prediction for 1 complex(es)...


Running run_protenix [00:00]

In [7]:
# Display output API reference
display_api_reference("protenix", "output", "run_protenix")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"Number of chains:  {len(complex.chains)}")
print(f"Protein length:    {len(mfng_sequence)} residues")
print(f"Confidence score:  {mfng_structure.metrics.confidence_score:.3f}")
print(f"Average pLDDT:     {mfng_structure.metrics.avg_plddt:.3f}")
print(f"pTM score:         {mfng_structure.metrics.ptm:.3f}")
print(f"ipTM score:        {mfng_structure.metrics.iptm:.3f}")
print(f"GPDE:              {mfng_structure.metrics.get('gpde', 'N/A')}")

# Visualize predicted complex colored by pLDDT confidence
mfng_structure.visualize(style="cartoon", color_by="bfactor")

**Output** — `ProtenixOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `List[Structure]` | required | Predicted structures with confidence metrics. |
| `structure` | `string` | required | Raw structure content in PDB or CIF format. |
| `structure_format` | `string` |  | Format of the content string (auto-detected if omitted). |
| `b_factor_type` | `BFactorType` |  | What the B-factor column represents. |
| `source` | `string` |  | Optional source identifier (filepath or tool name). |
| `metrics` | `StructureMetrics` |  | Associated metrics (e.g., pLDDT, pTM scores, per-chain lists, pairwise matrices). None values are stripped at construction. |

Number of chains:  2
Protein length:    384 residues
Confidence score:  0.937
Average pLDDT:     0.902
pTM score:         0.914
ipTM score:        0.943
GPDE:              0.4581643342971802


## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD, and as input to molecular dynamics or further design pipelines. The B-factor column contains pLDDT confidence scores for per-residue visualization in any compatible viewer.

In [8]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")